## ISOM4030 Lab 9 - Text Mining & Sentiment Analysis with RNN (Student Version)


In this lab, we will first demonstrate how the `TextVectorization` layer converts raw text into integer token sequences. We will then build several RNN models and train them on the [Yelp Polarity Reviews dataset](https://www.tensorflow.org/datasets/catalog/yelp_polarity_reviews) with word embeddings.

We want our model to be capable of differentiating sentiments expressed in these reviews even if the words used are exactly the same:

```
1: My friends do like the restaurant but I don't. --> negative review
2: My friends don't like the restaurant but I do. --> positive review
```



We will compare the performance and see which architecture might be best for this particular dataset. Let's begin!

 Firstly, import common libraries that will be used throughout the exercise.

In [ ]:
%pip install -q importlib-resources

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow.keras import layers, Sequential
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Text Vectorization with Keras

Neural networks cannot work directly with raw text. The `TextVectorization` layer performs the main preprocessing steps in one high-level API:

1. standardize and split the text into tokens;
2. build a vocabulary using `adapt()`;
3. map each token to an integer index; and
4. pad or truncate each sequence to a fixed length when `output_sequence_length` is specified.



In [ ]:
# A small text corpus for demonstration
sample_texts = tf.constant([
    "I love deep learning",
    "I love natural language processing",
    "Text mining is useful for business analytics"
])

# Keep at most 100 tokens and produce sequences of length 8
demo_vectorizer = layers.TextVectorization(
    max_tokens=100,
    output_mode="int",
    output_sequence_length=8
)

# Learn the vocabulary from the text corpus
demo_vectorizer.adapt(sample_texts)


In [ ]:
# Inspect the learned vocabulary
print("Vocabulary:", demo_vectorizer.get_vocabulary())

# Convert raw text into integer sequences
demo_sentences = tf.constant([
    "I love text mining",
    "Deep learning is useful"
])

encoded_sentences = demo_vectorizer(demo_sentences)
print("Encoded sentences:\n", encoded_sentences.numpy())


The vocabulary begins with two special entries: `''` is the padding token and `'[UNK]'` represents an out-of-vocabulary word. Each remaining word is assigned an integer index. Because `output_sequence_length=8`, shorter sentences are padded with zeros and longer sentences are truncated.

Next, we apply the same procedure to the Yelp Polarity Reviews dataset.


## 1 Download and Prepare the Dataset

Download the `yelp_polarity_reviews` dataset. It contains raw Yelp review text and binary sentiment labels. The full Yelp Polarity Reviews dataset contains 560,000 training reviews and 38,000 test reviews, with an equal number of positive and negative reviews in each split. For this lab, we use a balanced subset of 25,000 training reviews and 25,000 test reviews to reduce the computational cost and training time.

In [ ]:
# Load balanced subsets of Yelp reviews for binary sentiment classification
# Class 0 = negative sentiment; class 1 = positive sentiment
train_negative = tfds.load("yelp_polarity_reviews", split="train[:12500]", as_supervised=True)
train_positive = tfds.load("yelp_polarity_reviews", split="train[280000:292500]", as_supervised=True)
test_negative = tfds.load("yelp_polarity_reviews", split="test[:12500]", as_supervised=True)
test_positive = tfds.load("yelp_polarity_reviews", split="test[19000:31500]", as_supervised=True)

# Combine and shuffle the negative and positive subsets
train_data = train_negative.concatenate(train_positive).shuffle(25000, seed=42)
test_data = test_negative.concatenate(test_positive).shuffle(25000, seed=42)


In [ ]:
# Convert the tf.data.Dataset objects to Python lists and NumPy arrays
training_sentences, training_labels = [], []
testing_sentences, testing_labels = [], []

for text, label in tfds.as_numpy(train_data):
    training_sentences.append(text.decode("utf-8"))
    training_labels.append(label)
for text, label in tfds.as_numpy(test_data):
    testing_sentences.append(text.decode("utf-8"))
    testing_labels.append(label)

training_labels = np.array(training_labels)
testing_labels = np.array(testing_labels)
print(f"Training examples: {len(training_sentences):,}")
print(f"Test examples: {len(testing_sentences):,}")
print("Training label counts:", np.bincount(training_labels))
print("Test label counts:", np.bincount(testing_labels))


In [ ]:
# Display one negative Yelp review
negative_index = int(np.where(training_labels == 0)[0][0])
training_sentences[negative_index], training_labels[negative_index]


In [ ]:
# Display one positive Yelp review
positive_index = int(np.where(training_labels == 1)[0][0])
training_sentences[positive_index], training_labels[positive_index]



#### <font color='steelblue' > Exercise 1</font>
Now apply the `TextVectorization` workflow demonstrated above to the Yelp reviews. Create the layer, use `adapt()` to build the vocabulary from the training reviews, and convert both the training and testing reviews into fixed-length integer sequences.

In [ ]:
# parameters
vocab_size = 10000
max_length = 120

# Create a TextVectorization layer
# Fill in the arguments below.
text_vectorizer = layers.TextVectorization(
    max_tokens=____________________,
    output_mode=____________________,
    output_sequence_length=____________________
)

# Index the vocabulary via `adapt()`
text_vectorizer.____________________(training_sentences)

# Try out the layer
print(
    "Encoded text:\n", text_vectorizer([training_sentences[5]]).numpy(),
)

print(text_vectorizer.get_vocabulary()[:20])
text_vectorizer.vocabulary_size()


In [ ]:
# Vectorize the training and testing datasets
training_ds = ________________________________________
testing_ds = _________________________________________


In [ ]:
training_ds[:5, :]

To reverse the tokenization process and convert integer tokens back into their original strings, we use [**StringLookup**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/StringLookup).

StringLookup is a TensorFlow preprocessing layer that maps strings to integer indices and can invert this mapping when `invert=True`. It is widely used for handling categorical text data in machine learning workflows, especially when working with layers like TextVectorization.

Create an **inverted** StringLookup layer to map token IDs back to words.

- `vocabulary=vocab[2:]`: excludes special tokens ('' and "[UNK]").
- `mask_token=''` & `oov_token='[UNK]'`: ensure consistency with original tokenization.
- `invert=True`: enables reverse lookup (IDs → strings).

In [ ]:
from tensorflow.keras.layers import StringLookup
vocab = text_vectorizer.get_vocabulary()

# Create an inverted lookup that converts token IDs back to words.
token_to_string = StringLookup(
    vocabulary=____________________,
    mask_token=____________________,
    oov_token=____________________,
    invert=____________________
)


After creating the inverted StringLookup layer, you can pass token IDs to it to get back the original words. This will return a tensor of byte strings representing the words.

The **b prefix** means the string is stored as a bytes object in Python (common in TensorFlow for efficiency).



In [ ]:
token_to_string(training_ds[0])

---


## 2 Keras RNN APIs

### 2.1 Keras implementations of RNN Layers


There are three built-in RNN layers in Keras:

- [`SimpleRNN`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/SimpleRNN)

- [`LSTM`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM)

- [`GRU`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU)


```python
layers.SimpleRNN(hidden_units, activation='tanh', return_sequences=False, ...)
layers.LSTM(hidden_units, activation='tanh', recurrent_activation='sigmoid',  return_sequences=False, ...)
layers.GRU(hidden_units, activation='tanh', recurrent_activation='sigmoid', return_sequences=False, ...)

```

- `hidden_units` specifies the number of elements the hidden and the cell states have;

- `activation='tanh'` is the activation function used to update the hidden and cell states at each time step;

- `reccurent_activation='sigmoid'` is the activation function that generates the conceptual gates at each time step;

- There is no parameter for setting the number of time steps a model has, as Keras RNN APIs expand automatically along the time dimension to match the length of input sequences. So it's important to preprocess the input sequences to ensure all of them have a uniform length (e.g., each sequence was set to contain 120 tokens by `max_length = 120`).

- `return_sequences=False`: By default, a recurrent layer only return one output in the end. If you want to keep the outputs at all time steps (e.g., in the context of language modeling), set this to `True`.













## 2.2 Vanilla RNN Model

Now, we will build the 1st RNN model to take into account the temporal aspect of our input data (i.e., which word comes before or after which word).

All we need to do is simply replacing the `Flatten` layer above with a `SimpleRNN` layer.  Moreover, we will nest it inside a [Biderectional](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional) layer so as to encode the context information from both sides for each input sequence.



In [ ]:
# hyperparameters
embedding_dim = 100
state_dim = 64
dense_dim = 32

# Build a bidirectional vanilla RNN model.
model_vanilla = Sequential([
    layers.Input(shape=(max_length,), dtype='int64'),
    layers.Embedding(________________________________________),
    layers.Bidirectional(____________________________________),
    layers.Dense(____________________________________________),
    layers.Dense(____________________________________________)
], name="vanilla")

# Set the training parameters.
model_vanilla.compile(
    loss=____________________,
    optimizer=____________________,
    metrics=____________________
)

model_vanilla.summary()


Note that the output shape of the bidirectional layer is `(None, 128)`, indicating that the output vector is created by concatenating the output hidden states from both the forward and backward RNNs.


In [ ]:
# train the model
history_vanilla = model_vanilla.fit(training_ds, training_labels, batch_size=256, epochs=10, validation_data=(testing_ds, testing_labels))

In [ ]:
# plot the accuracy and loss history
metric_df = pd.DataFrame(history_vanilla.history)
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for metric, ax in zip(["loss", "accuracy"], axs):
  ax.plot(metric_df[metric], label=f'Training {metric}')
  ax.plot(metric_df['val_' + metric], label=f'Validation {metric}')
  ax.grid(True)
  ax.legend()

Run a prediction on a new sentence. If the prediction is greather than 0.5, it is positive else it is negative.

In [ ]:
sample_text_pos = ["""The restaurant was excellent. The staff were friendly and attentive,
the food was fresh and delicious, and the service was very efficient.
I would happily visit this place again."""]

sample_text_neg = ["""The restaurant was disappointing. The service was slow,
the staff were inattentive, and the food arrived cold and tasted bland.
I would not recommend this place."""]

sample_sequence_pos = text_vectorizer(sample_text_pos)
sample_sequence_neg = text_vectorizer(sample_text_neg)


In [ ]:
model_vanilla.predict(sample_sequence_neg)

## 2.3 LSTM Model

Next, we will use a *Long Short-Term Memory* model. A bidirectional RNN with LSTM blocks is usually a reasonable first thing to try. To faciliate model comparison, we use the same hyperparameters as before. You will notice in the model summary that the LSTM model has a larger number of paramenters than the vanilla model.


In [ ]:
# hyperparameters
embedding_dim = 100
state_dim = 64
dense_dim = 32

# Build a bidirectional LSTM model.
model_lstm = Sequential([
    layers.Input(shape=(max_length,), dtype='int64'),
    layers.Embedding(________________________________________),
    layers.Bidirectional(____________________________________),
    layers.Dense(____________________________________________),
    layers.Dense(____________________________________________)
], name="lstm")

model_lstm.compile(
    loss=____________________,
    optimizer=____________________,
    metrics=____________________
)

model_lstm.summary()


The output of the bidirectional layer has a shape of `(128,)`, implying that only the hidden states of the forward and backward RNNs will be passed to the rest of the model.

In [ ]:
# train the model
history_lstm = model_lstm.fit(training_ds, training_labels, batch_size=256, epochs=10, validation_data=(testing_ds, testing_labels))

In [ ]:
# plot the accuracy and loss history
metric_df = pd.DataFrame(history_lstm.history)
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for metric, ax in zip(["loss", "accuracy"], axs):
    ax.plot(metric_df[metric], label=f'Training {metric}')
    ax.plot(metric_df['val_' + metric], label=f'Validation {metric}')
    ax.grid(True)
    ax.legend()


In [ ]:
model_lstm.predict(sample_sequence_neg)

## 2.4 GRU Model

The *Gated Recurrent Unit* is usually referred to as a simpler version of the LSTM. It can be used when you want faster results and can sacrifice some accuracy.

In [ ]:
# hyperparameters
embedding_dim = 100
state_dim = 64
dense_dim = 32

# Build a bidirectional GRU model.
model_gru = Sequential([
    layers.Input(shape=(max_length,), dtype='int64'),
    layers.Embedding(________________________________________),
    layers.Bidirectional(____________________________________),
    layers.Dense(____________________________________________),
    layers.Dense(____________________________________________)
], name="gru")

model_gru.compile(
    loss=____________________,
    optimizer=____________________,
    metrics=____________________
)

model_gru.summary()


In [ ]:
# train the model
history_gru = model_gru.fit(training_ds, training_labels, batch_size=256, epochs=10, validation_data=(testing_ds, testing_labels))

In [ ]:
# plot the accuracy and loss history
metric_df = pd.DataFrame(history_gru.history)
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for metric, ax in zip(["loss", "accuracy"], axs):
    ax.plot(metric_df[metric], label=f'Training {metric}')
    ax.plot(metric_df['val_' + metric], label=f'Validation {metric}')
    ax.grid(True)
    ax.legend()


In [ ]:
model_gru.predict(sample_sequence_neg)

## 3 Multi-layer RNNs

We can build multiple layer LSTM models by simply appending `LSTM` layers in the `Sequential` container and enabling the `return_sequences` flag to `True`.  This is because an `LSTM` layer expects a sequence input so if the previous layer is also an LSTM, then it should output a sequence as well.



In [ ]:
# hyperparameters
embedding_dim = 100
state_dim = 64
dense_dim = 32

# Build a multi-layer bidirectional LSTM model.
# Remember that intermediate recurrent layers must return the full sequence.
model_multi_layer = Sequential([
    layers.Input(shape=(max_length,), dtype='int64'),
    layers.Embedding(________________________________________),
    layers.Bidirectional(layers.LSTM(state_dim, return_sequences=____________)),
    layers.Bidirectional(layers.LSTM(state_dim, return_sequences=____________)),
    layers.Bidirectional(layers.LSTM(state_dim)),
    layers.Dense(____________________________________________),
    layers.Dense(____________________________________________)
], name="multi_layer")

model_multi_layer.compile(
    loss=____________________,
    optimizer=____________________,
    metrics=____________________
)

model_multi_layer.summary()


Note that the outputs of both intermediate bidirectional RNNs have a shape of `(120, 128)`. It implies that the hidden states at 120 time steps have all been passed to the rest of the model when `return_sequences` is `True`.


In [ ]:
# train the model
history_multi_layer = model_multi_layer.fit(training_ds, training_labels, batch_size=256, epochs=10, validation_data=(testing_ds, testing_labels))

In [ ]:
# plot the accuracy and loss history
metric_df = pd.DataFrame(history_multi_layer.history)
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for metric, ax in zip(["loss", "accuracy"], axs):
    ax.plot(metric_df[metric], label=f'Training {metric}')
    ax.plot(metric_df['val_' + metric], label=f'Validation {metric}')
    ax.grid(True)
    ax.legend()


In [ ]:
model_multi_layer.predict(sample_sequence_neg)

## 4 RNNs with Pre-trained Word Embeddings

We will use GloVe (Global Vectors for Word Representation), a pre-trained word embedding method that represents words as dense numerical vectors based on word co-occurrence statistics from large text corpora.


The pre-trained GloVe model contains 100-dimensional vector representations for words learned from a large external text corpus. Instead of downloading and reading the GloVe text file manually, we can use **Gensim** to directly download and load the pre-trained word vectors.

**Gensim** is a Python library for natural language processing and vector-space modelling. We use Gensim's convenient Downloader API to access and load pre-trained GloVe word embeddings.

In [ ]:
%pip install -q gensim

In [ ]:
import gensim.downloader as api

embeddings_index = api.load("glove-wiki-gigaword-100")

The pre-trained GloVe model contains 100-dimensional word vectors for 400,000 distinct words. We use Gensim to directly download and load these pre-trained word vectors into **embeddings_index**, allowing us to easily retrieve the vector representation of a word.

In [ ]:
print(f"Found {len(embeddings_index)} word vectors.")
print(f"Number of dimensions: {embeddings_index['hi'].shape}")

We will now match the pre-trained GloVe word embeddings with the vocabulary created earlier by `TextVectorization`. The vocabulary is available through `text_vectorizer.get_vocabulary()`, so we can use it to construct the pre-trained embedding matrix.


In [ ]:
# Reuse the vocabulary learned by TextVectorization.
vocab = ______________________________________________
vocab_size_actual = __________________________________

# Create index-to-word and word-to-index lookups.
index_word = __________________________________________
word_index = __________________________________________

for i, word in enumerate(vocab[:10]):
    print(f'{i}: {word}')


In [ ]:
for word, idx in list(word_index.items())[:10]:
    print(f'{word}: {idx}')


The following code creates the embedding matrix with its $i$-th row containing the word vector for the $i$-th word in the dictionary:

In [ ]:
# Build an embedding matrix aligned with the TextVectorization vocabulary.
embedding_dim = 100
embedding_matrix = np.zeros((____________________, ____________________))

for i, word in enumerate(vocab):
    if i < 2:  # index 0 is padding, index 1 is [UNK]
        continue

    if word in embeddings_index:
        embedding_matrix[i] = ______________________________

print(embedding_matrix.shape)


Now, you can verify several rows in `embedding_matrix` with the corresponding entries in  `embeddings_index`:

In [ ]:
# index 0 is reserved for padding; index 1 represents [UNK]
idx = 2
word = index_word[idx]

print(f"The word associated with index {idx} is '{word}'")

if word in embeddings_index:
  print(embedding_matrix[idx, :] == embeddings_index[word])

In [ ]:
idx = 5
word = index_word[idx]

print(f"The word associated with index {idx} is '{word}'")

if word in embeddings_index:
  print(embedding_matrix[idx, :] == embeddings_index[word])

Now, to create a model that uses the pretrained glove embeddings instead of directly learning them from the training data, the [`Embedding` layer](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) should be defined as

```python
layers.Embedding(vocab_size_actual, embedding_dim, input_shape=(max_length,),
                 embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                 trainable=False)
```

or

```python
layers.Embedding(vocab_size_actual, embedding_dim, input_shape=(max_length,), weights=[embedding_matrix], trainable=False),
```
`trainable=False` ensures these vectors will be unchanged over the period of training.


This means that the GloVe word vectors are used as they are and will not be updated during training. Only the RNN and Dense layers will learn from the Yelp reviews.After training the model, compare the training accuracy and validation accuracy with the previous models.

In [ ]:
# hyperparameters
embedding_dim = 100
state_dim = 64
dense_dim = 32

# Build the multi-layer model using the pre-trained GloVe embedding matrix.
model_multi_glove = Sequential([
    layers.Embedding(
        ____________________,
        ____________________,
        input_shape=(max_length,),
        embeddings_initializer=________________________________________,
        trainable=____________
    ),
    layers.Bidirectional(layers.LSTM(state_dim, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(state_dim, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(state_dim)),
    layers.Dense(dense_dim, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name="multi_layer")

model_multi_glove.compile(
    loss=____________________,
    optimizer=____________________,
    metrics=____________________
)

model_multi_glove.summary()


In [ ]:
# train the model
history_multi_glove = model_multi_glove.fit(training_ds, training_labels, batch_size=256, epochs=10, validation_data=(testing_ds, testing_labels))

In [ ]:
# plot the accuracy and loss history
metric_df = pd.DataFrame(history_multi_glove.history)
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for metric, ax in zip(["loss", "accuracy"], axs):
  ax.plot(metric_df[metric], label=f'Training {metric}')
  ax.plot(metric_df['val_' + metric], label=f'Validation {metric}')
  ax.grid(True)
  ax.legend()

**Observe**: How well does the model perform when the pre-trained word embeddings are kept fixed? Is the result better or worse than the models that learned their own embeddings from the Yelp reviews?

**Your answer:**



In the previous CNN lab, we learned that a pre-trained CNN contains many layers. During fine-tuning, we normally keep most of the pre-trained network frozen and gradually unfreeze some of the higher layers. This helps preserve useful features learned by the pre-trained model.

GloVe is different. It does not consist of a stack of neural-network layers. Instead, the pre-trained GloVe embeddings are stored in a single embedding matrix, where each row represents the vector of a word. Therefore, there are no "higher GloVe layers" that we can selectively unfreeze.

Return to the model definition and change: `trainable=True`

**Observe**: How does the training and validation performance change after the GloVe embeddings become trainable? Why might fine-tuning the general-purpose GloVe vectors help with Yelp sentiment classification?

**Your answer:**

